# TP 1 - Ingénierie de consignes modèle


---
## 0. Configuration partagée


In [1]:
import json

from shared.config import ROOT_DIR
from shared.llm_utils import LLMRequest, run_llm
from shared.misc_utils import (
    find_and_parse_json_from_text,
    write_json_file,
)

LOG_DIR = ROOT_DIR / "TP1_travel_planner_LLM" / "logs"

user_query = """
Je veux partir 4 jours à Rome en avril, je n'ai pas encore les dates exactes.
Propose-moi un itinéraire de voyage. Mon budget est de 200 euros pour les sorties et les restaurants.
Je veux éviter les zones trop touristiques et découvrir des lieux plus confidentiels.
"""

#### Fonctions utilitaires pour estimer le coût et afficher l'usage des tokens

In [2]:
def token_estimate_cost_usd(
    token_usage: dict[str, int],
    input_price_per_1m_tokens_usd: float = 0.30,
    output_price_per_1m_tokens_usd: float = 2.50,
) -> float:
    input_cost = token_usage["input_tokens"] * input_price_per_1m_tokens_usd / 1_000_000
    output_cost = token_usage["output_tokens"] * output_price_per_1m_tokens_usd / 1_000_000
    return input_cost + output_cost


def token_print_report(token_usage: dict[str, int]) -> None:
    estimated_cost = token_estimate_cost_usd(token_usage)
    print(
        f"tokens : entrée={token_usage['input_tokens']} | "
        f"sortie={token_usage['output_tokens']} | "
        f"total={token_usage['total_tokens']}"
    )
    print(f"coût estimé (USD) : {estimated_cost:.6f}")


---
### Au préalable

On prépare une base technique pour la logique d'appel LLM

- `google_model_settings` : configuration partagée du modèle (température, top_p, top_k, max_tokens, budget de réflexion)
- `genai_client` : client Google GenAI authentifié, créé une seule fois
- `LLMRequest` **(TODO)** : classe qui représente les données d'entrée d'un appel LLM (`user_prompt` obligatoire, `system_prompt` optionnel)
- `LLMResponse` **(TODO)** : classe qui représente les données de sortie utiles (texte final, tokens, données brutes)
- `run_llm` **(TODO)** : fonction qui lit la configuration, envoie la requête au modèle et retourne un `LLMResponse`

Les fonctions et classes marquées TODO sont à implémenter dans `shared/llm_utils.py`.


---
## 1. Version 1: requête utilisateur seule


**TODO — Version V1**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`


Bloc de code qui construit la requête minimale avec `LLMRequest`, appelle `run_llm`, puis enregistre la sortie et les tokens dans `logs/llm_output_v1.txt`


In [3]:
# TODO : construire la requête minimale V1 sans system prompt
request_v1 = LLMRequest(
    system_prompt=None,
    user_prompt=user_query,
)
run_result_v1 = await run_llm(request_v1)
final_text_v1 = run_result_v1.output
token_usage_v1 = {
    "input_tokens": int(run_result_v1.input_tokens),
    "output_tokens": int(run_result_v1.output_tokens),
    "total_tokens": int(run_result_v1.total_tokens),
}

log_data_v1 = {
    "output_text": run_result_v1.output, "usage": token_usage_v1,
    "raw_response": run_result_v1.raw_response,
    "estimated_cost_usd": token_estimate_cost_usd(token_usage_v1)
}

log_path_v1 = LOG_DIR / "llm_output_v1.txt"
#print(f"toto : log_data_v1={log_data_v1}")
#write_json_file(file_path=log_path_v1, data=log_data_v1)

print(final_text_v1)

Super idée de partir à Rome ! Avec un budget de 200€ pour les sorties et les restaurants, et en cherchant des coins moins fréquentés, on peut faire un voyage mémorable. Voici une proposition d'itinéraire sur 4 jours, en privilégiant des lieux moins touristiques et en tenant compte de votre budget :

**Important :** Ce budget est serré pour Rome. Il faudra faire des choix avisés, privilégier les petits restaurants locaux, les marchés et cuisiner parfois pour économiser. Les prix indiqués sont des estimations et peuvent varier.

**Jour 1 : Découverte du Trastevere et de ses secrets**

* **Matin (Gratuit) :** Commencez votre voyage par le quartier du Trastevere. Perdez-vous dans ses ruelles pavées, admirez l'église de Santa Maria in Trastevere (l'une des plus anciennes églises de Rome) et le Ponte Sisto.
* **Déjeuner (10-15€) :** Trouvez une trattoria typique dans le Trastevere, loin de la place principale. Cherchez des menus "primi" (pâtes, risottos) pour des options abordables.
* **Aprè

In [9]:
token_print_report(token_usage_v1)


tokens : entrée=76 | sortie=1206 | total=1282
coût estimé (USD) : 0.003038


---
## 2. Version 2: prompt système structuré


**TODO — Version V2**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`

L'objectif est d'améliorer la qualité de la réponse avec des instructions claires.<br>
Pour cela, il faut définir :
- **Rôle** : ...
- **Contraintes** : ...
- **Structure** : 4 sections — résumé, itinéraire, budget, conseils.
- **Notes additionnelles** : *utilise cette section pour toute précision ou règle spéciale.*

**But :**
- Produire une réponse complète.
- Rester sous 3000 tokens.


In [4]:
# TODO : rédiger un system prompt contraint et réutilisable
system_prompt_v2 = """
## Instructions
Tu es un assistant expert en planification de voyage.
Réponds avec un texte structuré, clair et utile immédiatement.

Contraintes à respecter:
- Respecter la durée demandée.
- Respecter le budget indiqué.

Contraintes de style:
- Aller à l'essentiel, sans phrases inutiles.
- Pas de Markdown décoratif (gras, mise en forme complexe).
- Tu peux utiliser '##' pour séparer les sections.
- Pour chaque recommandation, donner 1 raison courte + 1 détail pratique concret.

Structure de réponse obligatoire:
1) Résumé
- Réponse directe en 2 à 4 phrases.
2) Itinéraire
- Plan jour par jour avec activités matin/après-midi.
- Inclure recommandations déjeuner/dîner.
3) Budget
- Estimation par catégorie (activités, repas, transport, extras).
4) Conseils pratiques
- Donner 3 conseils actionnables et pertinents.

Notes complémentaires:
- Si les dates sont flexibles, proposer la période la plus adaptée.
- Si le budget est serré, proposer une alternative moins chère pour chaque poste coûteux.
"""

request_v2 = LLMRequest(system_prompt=system_prompt_v2, user_prompt=user_query)
run_result_v2 = await run_llm(request_v2)
final_text_v2 = run_result_v2.output
token_usage_v2 = {
    "input_tokens": int(run_result_v2.input_tokens),
    "output_tokens": int(run_result_v2.output_tokens),
    "total_tokens": int(run_result_v2.total_tokens),
}

log_data_v2 = {"output_text": run_result_v2.output, "usage": token_usage_v2,
                  "raw_response": run_result_v2.raw_response,
                  "estimated_cost_usd": token_estimate_cost_usd(token_usage_v2)}

log_path_v2 = LOG_DIR / "llm_output_v2.txt"
#write_json_file(file_path=log_path_v2, data=log_data_v2)

print(final_text_v2)

## 1) Résumé

Voici une proposition d'itinéraire de 4 jours à Rome, axée sur la découverte de lieux moins fréquentés, respectant un budget de 200€ pour les sorties et les repas. L'itinéraire privilégie les quartiers plus authentiques et propose des expériences variées, allant de la culture à la gastronomie locale.  L'avril est une période idéale pour profiter de la ville sans les foules estivales.

## 2) Itinéraire

**Jour 1 : Trastevere et le Quartier Testaccio**

*   **Matin :** Exploration de Trastevere – Flânez dans les ruelles, admirez la basilique Santa Maria in Trastevere, et visitez le Ponte Sisto. *Raison : Ambiance typique romaine.* Détail : Déjeuner dans une trattoria locale à Trastevere (environ 15€).
*   **Après-midi :** Quartier Testaccio – Visitez le marché Testaccio, le Musée Nazionale Romano - Caffarella, et le Cimitero Acattolico (cimetière des non-catholiques). *Raison : Immersion dans la vie locale.* Détail : Dîner dans un restaurant typique à Testaccio (environ 20€

In [6]:
token_print_report(token_usage_v2)


tokens : entrée=322 | sortie=1765 | total=2776
coût estimé (USD) : 0.004509


---
## 3. Version 3: sortie structurée


**TODO — Version V3**

Fichier à modifier : `TP1_travel_planner_LLM/1_llm_assistant.ipynb`

L'objectif ici est d'avoir une sortie structurée, facile à parser et à traiter.
- Pour cela, il faut imposer un schéma strict JSON (ou XML, ou TOON), en plus de la réponse texte.
- Le prompt système final sera : <br>`system_prompt_v3 = system_prompt_v2 + structured_output_instructions`
- Il faut aussi créer une fonction de parsing pour extraire et parser le bloc JSON depuis le texte final.
- Enfin, on itérera sur ce JSON pour mesurer le coût total estimé et le comparer à la contrainte de budget.


In [5]:
# TODO : imposer la section JSON finale avec schéma strict
system_prompt_v3 = system_prompt_v2 + """

### Format de réponse
Tu dois d'abord répondre au format texte demandé.
Ensuite, ajoute un objet JSON valide contenant l'agenda détaillé.
Utilise exactement cette structure:
{
  "agenda": {
    "day_1": {
      "activity_am": {"title": "...", "address": "...", "estimated_cost_eur": ...},
      "activity_pm": {"title": "...", "address": "...", "estimated_cost_eur": ...},
      "lunch":       {"name": "...", "address": "...", "estimated_cost_eur": ...},
      "dinner":      {"name": "...", "address": "...", "estimated_cost_eur": ...}
    },
    "day_2": {...},
    "day_3": {...},
    "day_4": {...}
  }
}
"""

request_v3 = LLMRequest(system_prompt=system_prompt_v3, user_prompt=user_query)
run_result_v3 = await run_llm(request_v3)
final_text_v3 = run_result_v3.output
token_usage_v3 = {
    "input_tokens": int(run_result_v3.input_tokens),
    "output_tokens": int(run_result_v3.output_tokens),
    "total_tokens": int(run_result_v3.total_tokens),
}

log_data_v3 = {
    "output_text": run_result_v3.output,
    "usage": token_usage_v3,
    "raw_response": run_result_v3.raw_response,
}
log_data_v3["estimated_cost_usd"] = token_estimate_cost_usd(token_usage_v3)

log_path_v3 = LOG_DIR / "llm_output_v3.txt"
#write_json_file(file_path=log_path_v3, data=log_data_v3)

print(final_text_v3)

## Résumé

Voici une proposition d'itinéraire de 4 jours à Rome, axée sur la découverte de lieux moins fréquentés, respectant un budget de 200€ pour les activités et les repas. L'itinéraire est conçu pour être flexible en avril, avec des suggestions pour profiter au maximum de la ville.

## Itinéraire

**Jour 1 : Trastevere et le Quartier Coppedè**

*   **Matin:** Exploration de Trastevere : flânez dans les ruelles, visitez la Basilique Santa Maria in Trastevere. *Raison : Ambiance authentique et charmante.* Détail : Prenez un café dans une petite place pour observer la vie locale.
*   **Après-midi:** Quartier Coppedè : Visitez cette zone résidentielle unique, un exemple d'architecture Art Nouveau et Art Déco. *Raison : Un lieu architectural exceptionnel et peu connu.* Détail : Réservez une visite guidée pour en apprendre davantage sur l'histoire du quartier.
*   **Déjeuner:** Trattoria da Enzo al 29 (Trastevere) - 15€
*   **Dîner:** Roma Sparita (Trastevere) - 20€

**Jour 2 : Appia An

In [8]:
token_print_report(token_usage_v3)


tokens : entrée=514 | sortie=2382 | total=3647
coût estimé (USD) : 0.006109


#### Parser la sortie structurée

**TODO — `find_and_parse_json_from_text`**

Fichier à modifier : `shared/misc_utils.py`

`find_and_parse_json_from_text` : fonction qui récupère le dernier bloc JSON valide dans un texte de réponse, puis retourne un dictionnaire Python ou lève une erreur explicite


In [9]:
# TODO : parser la sortie texte+JSON sans post-traitement manuel
parsed_json_v3 = find_and_parse_json_from_text(final_text_v3)
print(json.dumps(obj=parsed_json_v3, ensure_ascii=False, indent=2))

{
  "agenda": {
    "day_1": {
      "activity_am": {
        "title": "Exploration du quartier Monti et église San Clemente",
        "address": "Via di San Giovanni in Laterano, 94, 00184 Roma RM",
        "estimated_cost_eur": 10
      },
      "activity_pm": {
        "title": "Balade extérieure Colisée et Forum Romain",
        "address": "Piazza del Colosseo, 1, 00184 Roma RM",
        "estimated_cost_eur": 0
      },
      "lunch": {
        "name": "Pizza al taglio dans Monti",
        "address": "Diverses adresses dans le quartier Monti",
        "estimated_cost_eur": 9
      },
      "dinner": {
        "name": "Trattoria typique à Monti",
        "address": "Ex: La Taverna dei Fori Imperiali, Via della Madonna dei Monti, 28, 00184 Roma RM",
        "estimated_cost_eur": 28
      }
    },
    "day_2": {
      "activity_am": {
        "title": "Quartier du Trastevere et vue depuis le Janicule",
        "address": "Piazzale Garibaldi, 00165 Roma RM",
        "estimated_cost_eur

#### Vérification du budget

In [10]:
agenda_v3 = parsed_json_v3["agenda"]

def compute_total_cost_from_json(agenda: dict[str, dict[str, dict[str, object]]]) -> float:
    total_cost = 0.0
    for day in agenda.values():
        for slot_name in ["activity_am", "activity_pm", "lunch", "dinner"]:
            total_cost += float(day[slot_name]["estimated_cost_eur"])
    return total_cost

total_cost_eur = compute_total_cost_from_json(agenda_v3)
average_per_day = total_cost_eur / len(agenda_v3)

print(f"Total estimé : {total_cost_eur:.2f} EUR")
print(f"Moyenne par jour : {average_per_day:.2f} EUR")


Total estimé : 130.00 EUR
Moyenne par jour : 32.50 EUR
